# 25a — Fisher uncertainty at truth vs KDE bandwidth

Series 25 · **no optimizer**. For each of the 14 experiments, simulate the
`(FWHM, sigma_fit)` cloud **once at the TRUE values**, then sweep the KDE bandwidth
(factors of Scott's rule, `h_f` and `h_s` scaled together) and read off the
Fisher/CRB uncertainty at the truth.

* Relative uncertainty σ/truth vs transmission, one line per bandwidth factor.
* μ and γ as stacked subplots; linear/log toggle; Scott (×1) marked thick.
* Hypothesis: the Fisher is bandwidth-sensitive through the 1/h² γ-score, so σ_γ
  should swing far more than σ_μ (whose score has no direct bandwidth term).


In [1]:
import os, sys, time
# locate repo root (for `import src`) from the src/fisher.py marker
for _up in ('', '..', '../..'):
    _p = os.path.abspath(_up)
    if os.path.exists(os.path.join(_p, 'src', 'fisher.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break

import numpy as np
import torch

from src.fisher import EXPERIMENTS, bandwidths, fisher_at, load_real_target, sim_cloud

M_FINAL = 500          # simulated scans per cloud (bump for final numbers)
SEED = 42
FACTORS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
torch.set_num_threads(1)
print('ready —', len(EXPERIMENTS), 'experiments,', len(FACTORS), 'bandwidth factors')

ready — 14 experiments, 7 bandwidth factors


In [2]:
results = []
t0 = time.time()
for (power, trans, mu_true, sigma_prop, lam, gamma_true) in EXPERIMENTS:
    target_f, target_s = load_real_target(int(power.replace('nW', '')), trans)
    cloud = sim_cloud(mu_true, sigma_prop, lam, gamma_true, M_FINAL, SEED)
    for factor in FACTORS:
        h_f, h_s = bandwidths(target_f, target_s, factor, factor, 0.05)
        fr = fisher_at(cloud, target_f, target_s, h_f, h_s, mu_true, sigma_prop,
                       gamma_scale=False, sigma_weight=1.0)
        results.append(dict(power=power, trans=trans, factor=factor, N=fr['N'],
                            mu_true=mu_true, gamma_true=gamma_true, h_f=h_f, h_s=h_s,
                            rel_mu=fr['sig_mu_data'] / mu_true,
                            rel_g=fr['sig_g_data'] / gamma_true))
    print(f"[{power} Trans{trans}] N={len(target_f)}  ({time.time()-t0:.0f}s)")

print(f"done in {time.time()-t0:.0f}s — {len(results)} rows")

[1nW Trans5] N=61  (20s)


[1nW Trans10] N=358  (38s)


[1nW Trans20] N=1138  (56s)


[1nW Trans40] N=2428  (75s)


[1nW Trans60] N=2424  (94s)


[1nW Trans80] N=2487  (113s)


... (4 outputs left out)

[3nW Trans40] N=3742  (211s)


[3nW Trans60] N=2541  (230s)


[3nW Trans80] N=2508  (250s)


[3nW Trans100] N=2516  (269s)
done in 269s — 98 rows


In [3]:
import plotly.graph_objects as go
import plotly.io as pio
import plotly.colors as pc
from plotly.subplots import make_subplots
from IPython.display import display

colors = pc.sample_colorscale('Viridis', np.linspace(0.05, 0.95, len(FACTORS)))
fcolor = dict(zip(FACTORS, colors))
trans_order = [5, 10, 20, 40, 60, 80, 100]

for power in ('1nW', '3nW'):
    rows = [r for r in results if r['power'] == power]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=('μ (relative uncertainty)', 'γ (relative uncertainty)'),
                        vertical_spacing=0.12)
    for factor in FACTORS:
        fr = sorted([r for r in rows if r['factor'] == factor],
                    key=lambda r: trans_order.index(r['trans']))
        xs = [r['trans'] for r in fr]
        scott = (factor == 1.0)
        fig.add_trace(go.Scatter(
            x=xs, y=[100 * r['rel_mu'] for r in fr], mode='lines+markers', name=f'×{factor}',
            line=dict(color=fcolor[factor], width=4 if scott else 2),
            marker=dict(size=9 if scott else 6), legendgroup=str(factor)), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=xs, y=[100 * r['rel_g'] for r in fr], mode='lines+markers', name=f'×{factor}',
            line=dict(color=fcolor[factor], width=4 if scott else 2),
            marker=dict(size=9 if scott else 6), legendgroup=str(factor), showlegend=False), row=2, col=1)
    fig.add_hline(y=10.0, line_dash='dot', line_color='grey', annotation_text='10%', row=1, col=1)
    fig.add_hline(y=10.0, line_dash='dot', line_color='grey', annotation_text='10%', row=2, col=1)
    fig.update_layout(title=f'Fisher uncertainty at truth vs KDE bandwidth — {power}',
                      height=800, legend_title='bandwidth (× Scott)',
                      hovermode='x unified', xaxis2_title='transmission (%)')
    fig.update_yaxes(type='linear', title_text='σ / truth (%)', row=1, col=1)
    fig.update_yaxes(type='linear', title_text='σ / truth (%)', row=2, col=1)
    fig.update_layout(updatemenus=[dict(type='buttons', direction='left', x=1.02, y=1.0, buttons=[
        dict(label='linear', method='relayout',
             args=[{'yaxis.type': 'linear', 'yaxis2.type': 'linear'}]),
        dict(label='log', method='relayout',
             args=[{'yaxis.type': 'log', 'yaxis2.type': 'log'}])])])
    display({'application/vnd.plotly.v1+json': pio.to_json(fig)}, raw=True)